In [3]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support


def avaliar_modelo(y_real, pontuacoes, limiar):
    y_previsto = (np.asarray(pontuacoes).reshape(-1) >= limiar).astype(int)

    precisao, recall, f1, _ = precision_recall_fscore_support(
        y_real, y_previsto, average="binary", pos_label=1, zero_division=0
    )
    matriz = confusion_matrix(y_real, y_previsto, labels=[0, 1])
    return {"precisao": precisao, "recall": recall, "f1": f1}, matriz


metricas, matriz = avaliar_modelo(
    y_real=[0, 0, 1, 1],
    pontuacoes=[0.10, 0.40, 0.80, 0.20],
    limiar=0.375,
)

print(metricas)
print(matriz)

{'precisao': 0.5, 'recall': 0.5, 'f1': 0.5}
[[1 1]
 [1 1]]


In [4]:
import json
from pathlib import Path

import joblib
import keras
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

artefatos = Path("artefatos")
config = json.loads(
    (artefatos / "configuracao_escolhida.json").read_text(encoding="utf-8")
)

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
_, df_teste = train_test_split(
    df,
    test_size=config["proporcao"]["teste_reservado"],
    random_state=config["seed"],
    stratify=df["Churn"],
)

X_teste = df_teste.loc[:, config["features_esperadas"]].copy()
X_teste["TotalCharges"] = pd.to_numeric(
    X_teste["TotalCharges"], errors="coerce"
).fillna(0.0)
y_teste = df_teste["Churn"].map({"No": 0, "Yes": 1}).to_numpy(dtype=np.int32)

preprocessador = joblib.load(artefatos / "preprocessador.joblib")
baseline = joblib.load(artefatos / "baseline_logistico.joblib")
rede = keras.models.load_model(artefatos / "modelo_churn.keras", compile=False)

X_teste_proc = np.asarray(preprocessador.transform(X_teste), dtype=np.float32)

comparacoes = [
    ("Regressão Logística", baseline.predict_proba(X_teste_proc)[:, 1], 0.375),
    (
        config["rede_selecionada_para_interface"],
        rede.predict(X_teste_proc, verbose=0).reshape(-1),
        config["limiar_rede"],
    ),
]

resultados = []
for nome, pontuacoes, limiar in comparacoes:
    metricas, matriz = avaliar_modelo(y_teste, pontuacoes, limiar)
    resultados.append({"modelo": nome, "limiar": limiar, **metricas})

    print(nome)
    display(pd.DataFrame(
        matriz,
        index=["Real: ficou", "Real: cancelou"],
        columns=["Previsto: ficou", "Previsto: cancelou"],
    ))

display(pd.DataFrame(resultados).round(4))

Regressão Logística


,Previsto: ficou,Previsto: cancelou
Real: ficou,832,203
Real: cancelou,116,258


Keras_1


,Previsto: ficou,Previsto: cancelou
Real: ficou,777,258
Real: cancelou,93,281


,modelo,limiar,precisao,recall,f1
0,Regressão Logística,0.375,0.5597,0.6898,0.6180
1,Keras_1,0.275,0.5213,0.7513,0.6156


## Conclusão de negócio

No conjunto de teste, a rede Keras_1 identificou 281 dos 374 clientes
que cancelaram (recall de 75,13%), enquanto a regressão logística
identificou 258 (68,98%). Se cada sinalização gerasse um contato de
retenção, a rede geraria 78 contatos adicionais: 23 com clientes que
cancelaram e 55 com clientes que permaneceram.

A regressão logística obteve maior precisão (55,97% contra 52,13%)
e um F1 ligeiramente superior (61,80% contra 61,56%). Para uma equipe
com capacidade limitada de contato, a regressão logística entregou
mais cancelamentos identificados por contato. Se a prioridade for
deixar passar menos cancelamentos e houver capacidade para mais
contatos, a Keras_1 apresentou maior cobertura neste teste.